# Excalibur Gen10 Brakes - Workbook-Based Combined Analysis

Unified Jupyter notebook for the four Excalibur Gen10 reference workbooks in `c:\Users\tdxt\OneDrive\Documents\GitHub\Gen11 Brakes\excalibur gen10 ref\calcs`:
- `Brake Force Calculator.xlsx`
- `Brake Pedal FBD Calculator.xlsx`
- `GenX CG Calculator.xlsx`
- `Rotor Thermal.xlsx`

This follows the same report style as the Gen11 combined notebook, but reads Gen10 workbook outputs directly.

Run all cells top-to-bottom after editing the reference workbooks.

Known workbook mismatches are called out explicitly instead of being hidden:
- `Brake Force Calculator.xlsx` and `Rotor Thermal.xlsx` use vehicle mass about `300 kg`
- `GenX CG Calculator.xlsx` uses vehicle mass `270 kg`
- `Brake Force Calculator.xlsx` uses pedal ratio `6.0`
- `Brake Pedal FBD Calculator.xlsx` geometry evaluates to pedal ratio `4.0`


In [ ]:
import math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from openpyxl import load_workbook

g = 9.81

def find_base_dir():
    candidates = [
        Path.cwd(),
        Path(r"c:\Users\tdxt\OneDrive\Documents\GitHub\Gen11 Brakes\excalibur gen10 ref\calcs"),
    ]
    for candidate in candidates:
        if (candidate / "Brake Force Calculator.xlsx").exists():
            return candidate
    raise FileNotFoundError("Could not find the Excalibur Gen10 calc workbooks.")

def row_values(ws, row_index, start_col, end_col):
    return [ws.cell(row_index, col).value for col in range(start_col, end_col + 1)]

def numeric_prefix(values):
    cleaned = []
    for value in values:
        if value in (None, ""):
            break
        cleaned.append(float(value))
    return cleaned

def text_prefix(values):
    cleaned = []
    for value in values:
        if value in (None, ""):
            break
        cleaned.append(str(value))
    return cleaned

def nearest_index(xs, target):
    xs = list(xs)
    return min(range(len(xs)), key=lambda idx: abs(xs[idx] - target))

def nearest_value(xs, ys, target):
    idx = nearest_index(xs, target)
    return xs[idx], ys[idx]

BASE_DIR = find_base_dir()
print("imports ok - g =", g)
print("using workbook directory:", BASE_DIR)


## Shared Workbook Inputs

Read the Gen10 workbooks and summarize the parameters that drive the rest of the notebook.


In [ ]:
wb_brake = load_workbook(BASE_DIR / "Brake Force Calculator.xlsx", data_only=True)
ws_brake_force = wb_brake["Brake Force Calculator"]
ws_weight = wb_brake["Weight Distribution"]

wb_fbd = load_workbook(BASE_DIR / "Brake Pedal FBD Calculator.xlsx", data_only=True)
ws_fbd = wb_fbd.active

wb_cg = load_workbook(BASE_DIR / "GenX CG Calculator.xlsx", data_only=True)
ws_cg = wb_cg["GenX CG"]

wb_thermal = load_workbook(BASE_DIR / "Rotor Thermal.xlsx", data_only=True)
ws_thermal_inputs = wb_thermal["Input Parameters"]
ws_thermal_stop = wb_thermal["Maximum Speed Emergency Stop"]
ws_thermal_repeat = wb_thermal["Level Ground Repeated Stops"]
ws_thermal_downhill = wb_thermal["Downhill Grade"]

brake_mass_kg = float(ws_brake_force["B2"].value)
tire_radius_m = float(ws_brake_force["B3"].value)
front_rotor_radius_m = float(ws_brake_force["B4"].value)
rear_rotor_radius_m = float(ws_brake_force["B5"].value)
mu_front_pad = float(ws_brake_force["B6"].value)
mu_rear_pad = float(ws_brake_force["B7"].value)
pedal_ratio_brake = float(ws_brake_force["B8"].value)
balance_bar_front = float(ws_brake_force["B9"].value)
tire_mu = float(ws_brake_force["B10"].value)

weight_total_N = float(ws_weight["B5"].value)
wheelbase_m = float(ws_weight["B3"].value)
cg_height_m = float(ws_weight["B7"].value)
static_front_weight_N = float(ws_weight["B8"].value)
static_rear_weight_N = float(ws_weight["B9"].value)

cg_mass_kg = float(ws_cg["B4"].value)

fbd_LA = float(ws_fbd["B5"].value)
fbd_LB = float(ws_fbd["B6"].value)
fbd_LC = float(ws_fbd["B7"].value)
fbd_theta_deg = float(ws_fbd["B8"].value)
fbd_theta_rad = float(ws_fbd["B9"].value)
fbd_pedal_ratio = float(ws_fbd["B10"].value)
fbd_foot_force_N = float(ws_fbd["D5"].value)
fbd_master_cylinder_force_N = float(ws_fbd["D17"].value)
fbd_pivot_force_N = float(ws_fbd["D11"].value)

thermal_mass_kg = float(ws_thermal_inputs["B6"].value)
thermal_h_W_m2K = float(ws_thermal_inputs["B14"].value)
thermal_rotor_outer_radius_m = float(ws_thermal_inputs["B10"].value)
thermal_rotor_inner_radius_m = float(ws_thermal_inputs["B11"].value)
thermal_rotor_thickness_m = float(ws_thermal_inputs["B12"].value)
thermal_rotor_surface_area_m2 = float(ws_thermal_inputs["E12"].value)
thermal_rho_kg_m3 = float(ws_thermal_inputs["B20"].value)
thermal_cp_J_kgK = float(ws_thermal_inputs["B18"].value)
thermal_ambient_K = float(ws_thermal_inputs["B24"].value)

print("--- Brake Force Calculator.xlsx ---")
print(f"mass = {brake_mass_kg:.1f} kg | wheelbase = {wheelbase_m:.3f} m | cg height = {cg_height_m:.3f} m")
print(f"tire radius = {tire_radius_m:.3f} m | front rotor radius = {front_rotor_radius_m:.5f} m | rear rotor radius = {rear_rotor_radius_m:.5f} m")
print(f"pad mu front/rear = {mu_front_pad:.2f} / {mu_rear_pad:.2f} | pedal ratio = {pedal_ratio_brake:.2f} | balance bar front = {balance_bar_front:.2f}")
print(f"static front/rear = {static_front_weight_N:.1f} N / {static_rear_weight_N:.1f} N ({100 * static_front_weight_N / weight_total_N:.1f}% front)")

print("\n--- GenX CG Calculator.xlsx ---")
print(f"standalone CG mass = {cg_mass_kg:.1f} kg")

print("\n--- Brake Pedal FBD Calculator.xlsx ---")
print(f"LA/LB/LC = {fbd_LA:.2f} / {fbd_LB:.2f} / {fbd_LC:.2f} | theta = {fbd_theta_deg:.1f} deg | pedal ratio = {fbd_pedal_ratio:.2f}")
print(f"foot force = {fbd_foot_force_N:.1f} N | master cylinder force = {fbd_master_cylinder_force_N:.1f} N | pivot force = {fbd_pivot_force_N:.1f} N")

print("\n--- Rotor Thermal.xlsx ---")
print(f"thermal mass input = {thermal_mass_kg:.1f} kg | h = {thermal_h_W_m2K:.1f} W/m^2K | ambient = {thermal_ambient_K:.1f} K")
print(f"rotor geometry: Ro = {thermal_rotor_outer_radius_m:.4f} m | Ri = {thermal_rotor_inner_radius_m:.4f} m | thickness = {thermal_rotor_thickness_m:.5f} m | area = {thermal_rotor_surface_area_m2:.5f} m^2")

unique_masses = {round(value, 3) for value in (brake_mass_kg, cg_mass_kg, thermal_mass_kg)}
if len(unique_masses) > 1:
    print("\nWARNING: workbook masses do not match exactly ->", sorted(unique_masses))
if abs(pedal_ratio_brake - fbd_pedal_ratio) > 1e-9:
    print(f"WARNING: brake workbook pedal ratio = {pedal_ratio_brake:.2f}, pedal FBD ratio = {fbd_pedal_ratio:.2f}")


## Dynamic Load Transfer

The Gen10 references include two CG-based sources. The brake workbook uses a `300 kg` vehicle, while the standalone CG workbook uses `270 kg`.
Both are plotted below so the mismatch stays visible.


In [ ]:
decels_weight = numeric_prefix(row_values(ws_weight, 12, 2, 14))
front_weight_300_N = numeric_prefix(row_values(ws_weight, 14, 2, 14))
rear_weight_300_N = numeric_prefix(row_values(ws_weight, 15, 2, 14))
front_bias_300_pct = [100.0 * value / weight_total_N for value in front_weight_300_N]

decels_cg = [float(ws_cg.cell(row, 1).value) for row in range(13, 19)]
front_weight_270_N = [float(ws_cg.cell(row, 3).value) for row in range(13, 19)]
rear_weight_270_N = [float(ws_cg.cell(row, 4).value) for row in range(13, 19)]
total_weight_270_N = float(ws_cg["B5"].value)
front_bias_270_pct = [100.0 * value / total_weight_270_N for value in front_weight_270_N]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(decels_weight, front_weight_300_N, "o-", label="Front axle - 300 kg workbook")
axes[0].plot(decels_weight, rear_weight_300_N, "o-", label="Rear axle - 300 kg workbook")
axes[0].plot(decels_cg, front_weight_270_N, "s--", label="Front axle - 270 kg workbook")
axes[0].plot(decels_cg, rear_weight_270_N, "s--", label="Rear axle - 270 kg workbook")
axes[0].set_xlabel("Deceleration [m/s^2]")
axes[0].set_ylabel("Axle load [N]")
axes[0].set_title("Dynamic axle loads")
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].plot(decels_weight, front_bias_300_pct, "o-", label="300 kg workbook")
axes[1].plot(decels_cg, front_bias_270_pct, "s--", label="270 kg workbook")
axes[1].set_xlabel("Deceleration [m/s^2]")
axes[1].set_ylabel("Front bias [% of total weight]")
axes[1].set_title("Front weight bias under braking")
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.0f}%"))
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8)

fig.suptitle("Excalibur Gen10 - CG and weight transfer references", fontsize=12)
plt.tight_layout()
plt.show()

for target in (4.72, 5.5, 9.81):
    x_300, front_300 = nearest_value(decels_weight, front_weight_300_N, target)
    _, rear_300 = nearest_value(decels_weight, rear_weight_300_N, target)
    x_270, front_270 = nearest_value(decels_cg, front_weight_270_N, target)
    _, rear_270 = nearest_value(decels_cg, rear_weight_270_N, target)
    print(
        f"{target:>4.2f} m/s^2 -> 300 kg workbook: front {front_300:,.1f} N, rear {rear_300:,.1f} N | "
        f"270 kg workbook ({x_270:>4.2f}): front {front_270:,.1f} N, rear {rear_270:,.1f} N"
    )


## Brake Force Calculator Outputs

Use the populated `Brake Force Calculator.xlsx` deceleration sweep to inspect brake split, pedal effort, and tire lock-up margin.


In [ ]:
decels_brake = numeric_prefix(row_values(ws_brake_force, 12, 2, 19))
total_force_N = numeric_prefix(row_values(ws_brake_force, 13, 2, 19))
front_force_total_N = numeric_prefix(row_values(ws_brake_force, 14, 2, 19))
rear_force_total_N = numeric_prefix(row_values(ws_brake_force, 15, 2, 19))
front_force_per_tire_N = numeric_prefix(row_values(ws_brake_force, 16, 2, 19))
rear_force_per_tire_N = numeric_prefix(row_values(ws_brake_force, 17, 2, 19))
pedal_force_N = numeric_prefix(row_values(ws_brake_force, 26, 2, 19))
pedal_force_lb = numeric_prefix(row_values(ws_brake_force, 27, 2, 19))
front_lock = text_prefix(row_values(ws_brake_force, 29, 2, 19))
rear_lock = text_prefix(row_values(ws_brake_force, 30, 2, 19))
front_available_tire_force_N = numeric_prefix(row_values(ws_brake_force, 31, 2, 19))
rear_available_tire_force_N = numeric_prefix(row_values(ws_brake_force, 32, 2, 19))

decels_lock = decels_brake[:len(front_lock)]
decels_front_limit = decels_brake[:len(front_available_tire_force_N)]
decels_rear_limit = decels_brake[:len(rear_available_tire_force_N)]

front_lock_decel = next((x for x, state in zip(decels_lock, front_lock) if state == "Yes"), None)
rear_lock_decel = next((x for x, state in zip(decels_lock, rear_lock) if state == "Yes"), None)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(decels_brake, front_force_total_N, "o-", label="Front axle demand")
axes[0, 0].plot(decels_brake, rear_force_total_N, "o-", label="Rear axle demand")
axes[0, 0].set_title("Brake force demand by axle")
axes[0, 0].set_xlabel("Deceleration [m/s^2]")
axes[0, 0].set_ylabel("Force [N]")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(decels_brake, pedal_force_N, "o-", color="#d62728", label="Pedal force [N]")
axes[0, 1].set_title("Pedal effort")
axes[0, 1].set_xlabel("Deceleration [m/s^2]")
axes[0, 1].set_ylabel("Pedal force [N]", color="#d62728")
axes[0, 1].tick_params(axis="y", labelcolor="#d62728")
axes01b = axes[0, 1].twinx()
axes01b.plot(decels_brake, pedal_force_lb, "s--", color="#1f77b4", label="Pedal force [lb]")
axes01b.set_ylabel("Pedal force [lb]", color="#1f77b4")
axes01b.tick_params(axis="y", labelcolor="#1f77b4")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(decels_front_limit, front_force_per_tire_N[:len(decels_front_limit)], "o-", label="Demanded front tire force")
axes[1, 0].plot(decels_front_limit, front_available_tire_force_N, "s--", label="Available front tire force")
if front_lock_decel is not None:
    axes[1, 0].axvline(front_lock_decel, color="k", linestyle=":", alpha=0.7, label=f"Front lock starts @ {front_lock_decel:.2f}")
axes[1, 0].set_title("Front tire lock-up margin")
axes[1, 0].set_xlabel("Deceleration [m/s^2]")
axes[1, 0].set_ylabel("Per-tire force [N]")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend(fontsize=8)

axes[1, 1].plot(decels_rear_limit, rear_force_per_tire_N[:len(decels_rear_limit)], "o-", label="Demanded rear tire force")
axes[1, 1].plot(decels_rear_limit, rear_available_tire_force_N, "s--", label="Available rear tire force")
if rear_lock_decel is not None:
    axes[1, 1].axvline(rear_lock_decel, color="k", linestyle=":", alpha=0.7, label=f"Rear lock starts @ {rear_lock_decel:.2f}")
axes[1, 1].set_title("Rear tire lock-up margin")
axes[1, 1].set_xlabel("Deceleration [m/s^2]")
axes[1, 1].set_ylabel("Per-tire force [N]")
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend(fontsize=8)

fig.suptitle("Excalibur Gen10 - Brake force workbook sweep", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Front tires first lock at about {front_lock_decel:.2f} m/s^2" if front_lock_decel is not None else "Front tires do not lock in the populated sweep.")
print(f"Rear tires first lock at about {rear_lock_decel:.2f} m/s^2" if rear_lock_decel is not None else "Rear tires do not lock in the populated sweep.")

for target in (4.72, 5.5, 9.81):
    idx = nearest_index(decels_brake, target)
    front_bias_pct = 100.0 * front_force_total_N[idx] / total_force_N[idx]
    print(
        f"{decels_brake[idx]:>4.2f} m/s^2 -> total {total_force_N[idx]:,.1f} N | "
        f"front {front_force_total_N[idx]:,.1f} N ({front_bias_pct:.1f}%) | "
        f"rear {rear_force_total_N[idx]:,.1f} N | pedal {pedal_force_N[idx]:,.1f} N ({pedal_force_lb[idx]:.1f} lb)"
    )


## Pedal FBD Geometry

The free-body workbook resolves the pedal linkage for an `890 N` foot input.
This is a geometry check, not the same pedal-ratio assumption used in the brake force workbook.


In [ ]:
foot_x_N = fbd_foot_force_N * math.cos(fbd_theta_rad)
foot_y_N = fbd_foot_force_N * math.sin(fbd_theta_rad)
mc_force_calc_N = ((foot_x_N * fbd_LB) + (foot_y_N * fbd_LC)) / fbd_LA
pivot_x_N = mc_force_calc_N + foot_x_N
pivot_y_N = foot_y_N
pivot_force_calc_N = math.hypot(pivot_x_N, pivot_y_N)
beta_deg = math.degrees(math.atan2(pivot_y_N, pivot_x_N))

target_labels = ["4.72", "5.5", "9.81", "FBD input"]
target_pedal_forces_N = [
    pedal_force_N[nearest_index(decels_brake, 4.72)],
    pedal_force_N[nearest_index(decels_brake, 5.5)],
    pedal_force_N[nearest_index(decels_brake, 9.81)],
    fbd_foot_force_N,
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(
    ["Foot x", "Foot y", "MC x", "Pivot"],
    [foot_x_N, foot_y_N, mc_force_calc_N, pivot_force_calc_N],
    color=["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"],
)
axes[0].set_title("Pedal FBD force components")
axes[0].set_ylabel("Force [N]")
axes[0].grid(True, axis="y", alpha=0.3)

axes[1].bar(target_labels, target_pedal_forces_N, color=["#1f77b4", "#1f77b4", "#1f77b4", "#d62728"])
axes[1].set_title("Brake-workbook pedal demand vs FBD input")
axes[1].set_ylabel("Pedal force [N]")
axes[1].grid(True, axis="y", alpha=0.3)

fig.suptitle("Excalibur Gen10 - Pedal force consistency check", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Foot force components: x = {foot_x_N:.1f} N, y = {foot_y_N:.1f} N")
print(f"Computed master-cylinder force = {mc_force_calc_N:.1f} N (workbook value {fbd_master_cylinder_force_N:.1f} N)")
print(f"Computed pivot force = {pivot_force_calc_N:.1f} N at beta = {beta_deg:.2f} deg (workbook value {fbd_pivot_force_N:.1f} N)")
print(f"Pedal ratio comparison -> brake workbook {pedal_ratio_brake:.2f}, pedal FBD workbook {fbd_pedal_ratio:.2f}")
print(f"FBD input foot force exceeds the brake-workbook 9.81 m/s^2 pedal demand by {fbd_foot_force_N - pedal_force_N[nearest_index(decels_brake, 9.81)]:.1f} N")


## Rotor Thermal Reference

`Rotor Thermal.xlsx` only has populated numbers in the maximum-speed emergency-stop block.
The repeated-stops and downhill sheets are empty or incomplete, so this cell summarizes the usable emergency-stop reference and flags the missing sections.


In [ ]:
thermal_total_weight_N = float(ws_thermal_stop["B2"].value)
thermal_stop_mass_kg = float(ws_thermal_stop["B3"].value)
thermal_front_rotor_mass_kg = float(ws_thermal_stop["B4"].value)
thermal_rear_rotor_mass_kg = float(ws_thermal_stop["B5"].value)
thermal_stop_cp_J_kgK = float(ws_thermal_stop["B6"].value)
thermal_speed_mps = float(ws_thermal_stop["B7"].value)
thermal_vehicle_ke_kJ = float(ws_thermal_stop["B8"].value)
thermal_stop_decel_mps2 = float(ws_thermal_stop["B10"].value)
thermal_stop_time_s = float(ws_thermal_stop["B11"].value)

thermal_front_weight_frac = float(ws_thermal_stop["E3"].value)
thermal_rear_weight_frac = float(ws_thermal_stop["E4"].value)
thermal_right_weight_frac = float(ws_thermal_stop["E5"].value)
thermal_total_power_kW = float(ws_thermal_stop["E8"].value)
thermal_front_right_power_kW = float(ws_thermal_stop["E9"].value)
thermal_rear_right_power_kW = float(ws_thermal_stop["E10"].value)
thermal_front_right_deltaT_K = float(ws_thermal_stop["E11"].value)
thermal_rear_right_deltaT_K = float(ws_thermal_stop["E12"].value)
thermal_front_final_K = float(ws_thermal_stop["E13"].value)
thermal_rear_final_K = float(ws_thermal_stop["E14"].value)
thermal_front_energy_kJ = float(ws_thermal_stop["E16"].value)
thermal_rear_energy_kJ = float(ws_thermal_stop["E17"].value)

repeated_stops_empty = ws_thermal_repeat.max_row == 1 and ws_thermal_repeat.max_column == 1 and ws_thermal_repeat["A1"].value is None
downhill_sheet_empty = ws_thermal_downhill.max_row == 1 and ws_thermal_downhill.max_column == 1 and ws_thermal_downhill["A1"].value is None

front_right_energy_recalc_kJ = thermal_front_right_power_kW * thermal_stop_time_s
rear_right_energy_recalc_kJ = thermal_rear_right_power_kW * thermal_stop_time_s

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

axes[0].bar(["Front right", "Rear right"], [thermal_front_right_power_kW, thermal_rear_right_power_kW], color=["#1f77b4", "#ff7f0e"])
axes[0].set_title("Emergency-stop braking power")
axes[0].set_ylabel("Power [kW]")
axes[0].grid(True, axis="y", alpha=0.3)

axes[1].bar(["Front right", "Rear right"], [thermal_front_energy_kJ, thermal_rear_energy_kJ], color=["#1f77b4", "#ff7f0e"])
axes[1].set_title("Energy into each rotor")
axes[1].set_ylabel("Energy [kJ]")
axes[1].grid(True, axis="y", alpha=0.3)

axes[2].bar(["Front right", "Rear right"], [thermal_front_right_deltaT_K, thermal_rear_right_deltaT_K], color=["#1f77b4", "#ff7f0e"])
axes[2].set_title("Rotor temperature rise")
axes[2].set_ylabel("Delta T [K]")
axes[2].grid(True, axis="y", alpha=0.3)

fig.suptitle("Excalibur Gen10 - Rotor thermal workbook reference", fontsize=12)
plt.tight_layout()
plt.show()

print("--- Maximum Speed Emergency Stop ---")
print(f"vehicle mass = {thermal_stop_mass_kg:.1f} kg | speed = {thermal_speed_mps:.1f} m/s | KE = {thermal_vehicle_ke_kJ:.1f} kJ")
print(f"decel = {thermal_stop_decel_mps2:.2f} m/s^2 | stop time = {thermal_stop_time_s:.2f} s | total braking power = {thermal_total_power_kW:.3f} kW")
print(f"front-right rotor: {thermal_front_right_power_kW:.3f} kW, {thermal_front_energy_kJ:.3f} kJ, delta T = {thermal_front_right_deltaT_K:.2f} K, final T = {thermal_front_final_K:.2f} K")
print(f"rear-right rotor:  {thermal_rear_right_power_kW:.3f} kW, {thermal_rear_energy_kJ:.3f} kJ, delta T = {thermal_rear_right_deltaT_K:.2f} K, final T = {thermal_rear_final_K:.2f} K")
print(f"energy cross-check from power*time -> front {front_right_energy_recalc_kJ:.3f} kJ, rear {rear_right_energy_recalc_kJ:.3f} kJ")
print(f"thermal inputs -> h = {thermal_h_W_m2K:.1f} W/m^2K, cp = {thermal_cp_J_kgK:.1f} J/kgK, rho = {thermal_rho_kg_m3:.1f} kg/m^3")

if repeated_stops_empty:
    print("NOTE: 'Level Ground Repeated Stops' sheet is empty in the reference workbook.")
if downhill_sheet_empty:
    print("NOTE: 'Downhill Grade' sheet is empty in the reference workbook.")
if repeated_stops_empty or downhill_sheet_empty:
    print("Use the populated emergency-stop block as the current thermal reference; downhill/repeated-stop studies still need workbook data.")
